<a href="https://colab.research.google.com/github/nimishavarma27/headline-relevance-classification-telugu/blob/main/teclass_ml_all_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TeClass: all ML baselines

4 feature setups from Table 3 of the paper, each with LR / SVM / MLP:

1. No features (just TF-IDF)
2. cosine similarity
3. cosine + LEAD-1 + novel 1-gram %
4. cosine + LEAD-1 + EXT-ORACLE + novel 1-gram % + novel 2-gram %

Uses a custom Telugu tokenizer (modified indic tokenizer) for sentence-splitting and word tokens for the features. For TF-IDF we use **SentencePiece BPE subwords** (vocab 2945). MLP uses early stopping.

## 1. Imports

In [ ]:
!pip install -q sentencepiece

In [ ]:
import re
import os
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

import sentencepiece as spm

SEED = 42
np.random.seed(SEED)

## 2. Telugu tokenizer

Rules:
- `..` or `...` do not end a sentence
- `?` does not end a sentence
- newline ends a sentence
- text inside `()` `[]` `{}` stays as one sentence
- numbers and dates stay as one token

This tokenizer is used for sentence splitting and word-level features (LEAD-1, EXT-ORACLE, novel n-gram). TF-IDF uses SentencePiece.

In [ ]:
class TeluguTokenizer:
    PUNCS = set('.,;:!?()[]{}"\'\u2018\u2019\u201c\u201d\u0964\u0965-\u2013\u2014')
    SENT_END = {'.', '\u0964', '\u0965'}
    OPEN_BR  = {'(', '[', '{'}
    CLOSE_BR = {')', ']', '}'}
    NUM_RE   = re.compile(r'\d+(?:[.,\-/:]\d+)*')

    def preprocess_data(self, text):
        text = text.replace('\t', ' ').replace('\u200b', '')
        text = re.sub(r'(\.{2,})"', r'\1 "', text)
        text = re.sub(r"(\.{2,})'", r"\1 '", text)
        text = re.sub(r'(\.{2,})-', r'\1 -', text)
        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'\r+', '\r', text)
        text = re.sub(r'[ \t]+', ' ', text)
        return text.strip()

    def trivial_tokenize_indic(self, text):
        nums = []
        def stash(m):
            nums.append(m.group(0))
            return f' \x01N{len(nums)-1}\x02 '
        text = self.NUM_RE.sub(stash, text)

        out, i = [], 0
        while i < len(text):
            c = text[i]
            if c in self.PUNCS:
                j = i + 1
                while j < len(text) and text[j] == c:
                    j += 1
                out.append(' ' + text[i:j] + ' ')
                i = j
            else:
                out.append(c)
                i += 1
        text = ''.join(out)

        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r' *([\n\r]) *', r'\1', text)

        for idx, n in enumerate(nums):
            text = text.replace(f'\x01N{idx}\x02', n)
        return text.strip()

    def sentence_tokenize(self, text):
        text = self.trivial_tokenize_indic(self.preprocess_data(text))
        sentences = []
        for line in re.split(r'[\n\r]+', text):
            line = line.strip()
            if not line:
                continue
            cur, depth = [], 0
            for tok in line.split(' '):
                if not tok:
                    continue
                cur.append(tok)
                if tok in self.OPEN_BR:
                    depth += 1
                elif tok in self.CLOSE_BR:
                    depth = max(0, depth - 1)
                elif tok in self.SENT_END and depth == 0:
                    sentences.append(' '.join(cur))
                    cur = []
            if cur:
                sentences.append(' '.join(cur))
        return sentences if sentences else [text]

    def word_tokenize(self, sentences):
        return [s.split() for s in sentences]

    def remove_punctuation(self, tokens):
        return [t for t in tokens if not all(c in self.PUNCS for c in t)]


tokenizer = TeluguTokenizer()

example = "మంతి? తానేటి వనిత సంతకం ఫోరజ్రీ చేశారు. కడప జిలా ల్కు చెందిన టీడీపీ నేత ఫోరజ్రీ చేశాడు."
for s in tokenizer.sentence_tokenize(example):
    print(s)

మంతి ? తానేటి వనిత సంతకం ఫోరజ్రీ చేశారు .
కడప జిలా ల్కు చెందిన టీడీపీ నేత ఫోరజ్రీ చేశాడు .


## 3. Load data

In [ ]:
TRAIN_PATH = "train.csv"
DEV_PATH   = "dev.csv"
TEST_PATH  = "test.csv"

def load(path):
    df = pd.read_csv(path)
    art_col  = next(c for c in df.columns if c.lower() in ["article", "text", "body", "news_article"])
    head_col = next(c for c in df.columns if c.lower() in ["headline", "title", "news_headline"])
    lab_col  = next(c for c in df.columns if c.lower() in ["label", "category", "class"])
    df = df[[art_col, head_col, lab_col]].copy()
    df.columns = ["article", "headline", "label"]
    df["label"] = df["label"].astype(str).str.upper().str.strip()
    df = df[df["label"].isin(["HREL", "MREL", "LREL"])].reset_index(drop=True)
    return df

train_df = load(TRAIN_PATH)
dev_df   = load(DEV_PATH)
test_df  = load(TEST_PATH)

LABELS = ["HREL", "MREL", "LREL"]
label2id = {"HREL": 0, "MREL": 1, "LREL": 2}

y_train = train_df["label"].map(label2id).values
y_dev   = dev_df["label"].map(label2id).values
y_test  = test_df["label"].map(label2id).values

print("train:", train_df.shape, " dev:", dev_df.shape, " test:", test_df.shape)
print(train_df["label"].value_counts())

train: (18324, 3)  dev: (3927, 3)  test: (3927, 3)
label
MREL    7105
HREL    5962
LREL    5257
Name: count, dtype: int64


## 4. SentencePiece subword tokenizer (for TF-IDF)

Train a BPE model on the training articles + headlines. Vocabulary size 2945 . This tokenizer is used only inside TF-IDF; the word-level features in Section 7 keep using the Telugu tokenizer.

In [ ]:
SP_VOCAB_SIZE = 2945
SP_CORPUS = "sp_corpus.txt"
SP_PREFIX = "sp"

# training corpus (articles + headlines from train only)
if not os.path.exists(SP_CORPUS):
    with open(SP_CORPUS, "w", encoding="utf-8") as f:
        for t in pd.concat([train_df["article"], train_df["headline"]]):
            f.write(str(t).replace("\n", " ") + "\n")

# train BPE model
if not os.path.exists(f"{SP_PREFIX}.model"):
    spm.SentencePieceTrainer.Train(
        input=SP_CORPUS,
        model_prefix=SP_PREFIX,
        vocab_size=SP_VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,
        shuffle_input_sentence=False,
    )

sp = spm.SentencePieceProcessor(model_file=f"{SP_PREFIX}.model")

def sp_tokenize(text):
    return sp.encode(str(text), out_type=str)

print("vocab size:", sp.get_piece_size())
print("example subword split:")
print(sp_tokenize("మంతి? తానేటి వనిత సంతకం ఫోరజ్రీ చేశారు")[:20])

vocab size: 2945
example subword split:
['▁మ', 'ంతి', '?', '▁తా', 'నే', 'టి', '▁వ', 'ని', 'త', '▁సం', 'త', 'కం', '▁ఫో', 'ర', 'జ', '్రీ', '▁చేశారు']


## 5. Tokenize once (word level, for features)

Word tokens and sentence splits are reused by LEAD-1, EXT-ORACLE and novel n-gram %.

In [ ]:
def to_tokens(text):
    sents = tokenizer.sentence_tokenize(text)
    words = [w for s in tokenizer.word_tokenize(sents) for w in s]
    return tokenizer.remove_punctuation(words)

print("tokenizing articles + headlines ...")
train_art_toks  = [to_tokens(t) for t in train_df["article"]]
train_head_toks = [to_tokens(t) for t in train_df["headline"]]
dev_art_toks    = [to_tokens(t) for t in dev_df["article"]]
dev_head_toks   = [to_tokens(t) for t in dev_df["headline"]]
test_art_toks   = [to_tokens(t) for t in test_df["article"]]
test_head_toks  = [to_tokens(t) for t in test_df["headline"]]

print("splitting articles into sentences ...")
train_art_sents = [tokenizer.sentence_tokenize(t) for t in train_df["article"]]
dev_art_sents   = [tokenizer.sentence_tokenize(t) for t in dev_df["article"]]
test_art_sents  = [tokenizer.sentence_tokenize(t) for t in test_df["article"]]

print("done")

tokenizing articles + headlines ...
splitting articles into sentences ...
done


## 6. TF-IDF vectorizers (using SentencePiece subwords)

One vectorizer for articles, one for headlines, and a shared vectorizer for cosine similarity. All three use the same SentencePiece tokenizer so the vocabulary is BPE subwords.

In [ ]:
def make_tfidf():
    return TfidfVectorizer(
        tokenizer=sp_tokenize,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        min_df=2,
        sublinear_tf=True,
    )

art_vec  = make_tfidf()
head_vec = make_tfidf()
sim_vec  = make_tfidf()

X_train_art  = art_vec.fit_transform(train_df["article"])
X_train_head = head_vec.fit_transform(train_df["headline"])
X_dev_art    = art_vec.transform(dev_df["article"])
X_dev_head   = head_vec.transform(dev_df["headline"])
X_test_art   = art_vec.transform(test_df["article"])
X_test_head  = head_vec.transform(test_df["headline"])

sim_vec.fit(pd.concat([train_df["article"], train_df["headline"]]))

X_train = hstack([X_train_art, X_train_head]).tocsr()
X_dev   = hstack([X_dev_art,   X_dev_head]).tocsr()
X_test  = hstack([X_test_art,  X_test_head]).tocsr()

print("X_train:", X_train.shape, " X_dev:", X_dev.shape, " X_test:", X_test.shape)
print("article vocab:", len(art_vec.vocabulary_), " headline vocab:", len(head_vec.vocabulary_))

X_train: (18324, 5637)  X_dev: (3927, 5637)  X_test: (3927, 5637)
article vocab: 2916  headline vocab: 2721


In [ ]:
# SentencePiece vocab size
print("SentencePiece vocab size:", sp.get_piece_size())

# Actual TF-IDF vocab sizes
print("Article TF-IDF vocab:", len(art_vec.vocabulary_))
print("Headline TF-IDF vocab:", len(head_vec.vocabulary_))
print("Shared TF-IDF vocab:", len(sim_vec.vocabulary_))

SentencePiece vocab size: 2945
Article TF-IDF vocab: 2916
Headline TF-IDF vocab: 2721
Shared TF-IDF vocab: 2918


## 7. Feature functions

Cosine similarity, ROUGE-L (for LEAD-1 and EXT-ORACLE), novel n-gram %.

In [ ]:
def cosine_sim(articles, headlines, vec):
    A = vec.transform(articles)
    B = vec.transform(headlines)
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()

def lcs(a, b):
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]

def rouge_l(ref, cand):
    if not ref or not cand:
        return 0.0
    l = lcs(ref, cand)
    if l == 0:
        return 0.0
    p = l / len(cand)
    r = l / len(ref)
    return 2 * p * r / (p + r)

def lead1(art_sents, head_toks):
    out = np.zeros(len(art_sents))
    for i, (sents, h) in enumerate(zip(art_sents, head_toks)):
        first = sents[0].split() if sents else []
        out[i] = rouge_l(first, h)
    return out

def ext_oracle(art_sents, head_toks):
    out = np.zeros(len(art_sents))
    for i, (sents, h) in enumerate(zip(art_sents, head_toks)):
        best = 0.0
        for s in sents:
            score = rouge_l(s.split(), h)
            if score > best:
                best = score
        out[i] = best
        if (i + 1) % 2000 == 0:
            print(f"  ext-oracle: {i+1}/{len(art_sents)}")
    return out

def novel_ngram(art_toks, head_toks, n):
    out = np.zeros(len(art_toks))
    for i, (a, h) in enumerate(zip(art_toks, head_toks)):
        h_ng = {tuple(h[k:k+n]) for k in range(len(h) - n + 1)}
        a_ng = {tuple(a[k:k+n]) for k in range(len(a) - n + 1)}
        out[i] = 0.0 if not h_ng else len(h_ng - a_ng) / len(h_ng)
    return out

## 8. Compute all features

LEAD-1, EXT-ORACLE and novel n-gram % are pure string features so they can be computed once. Cosine is recomputed inside each CV fold below to avoid leakage.

In [ ]:
print("cosine similarity ...")
cos_train = cosine_sim(train_df["article"], train_df["headline"], sim_vec)
cos_dev   = cosine_sim(dev_df["article"],   dev_df["headline"],   sim_vec)
cos_test  = cosine_sim(test_df["article"],  test_df["headline"],  sim_vec)
print("  done")

print("\nLEAD-1 ...")
lead1_train = lead1(train_art_sents, train_head_toks)
lead1_dev   = lead1(dev_art_sents,   dev_head_toks)
lead1_test  = lead1(test_art_sents,  test_head_toks)
print("  done")

print("\nnovel 1-gram % ...")
nov1_train = novel_ngram(train_art_toks, train_head_toks, 1)
nov1_dev   = novel_ngram(dev_art_toks,   dev_head_toks,   1)
nov1_test  = novel_ngram(test_art_toks,  test_head_toks,  1)
print("  done")

print("\nnovel 2-gram % ...")
nov2_train = novel_ngram(train_art_toks, train_head_toks, 2)
nov2_dev   = novel_ngram(dev_art_toks,   dev_head_toks,   2)
nov2_test  = novel_ngram(test_art_toks,  test_head_toks,  2)
print("  done")

print("\nEXT-ORACLE (slow, takes a few minutes) ...")
print("  computing for TRAIN split ({} rows) ...".format(len(train_art_sents)))
ext_train = ext_oracle(train_art_sents, train_head_toks)
print("  train done.")
print("  computing for DEV split ({} rows) ...".format(len(dev_art_sents)))
ext_dev = ext_oracle(dev_art_sents, dev_head_toks)
print("  dev done.")
print("  computing for TEST split ({} rows) ...".format(len(test_art_sents)))
ext_test = ext_oracle(test_art_sents, test_head_toks)
print("  test done.")

print("\nAll features computed.")

cosine similarity ...
  done

LEAD-1 ...
  done

novel 1-gram % ...
  done

novel 2-gram % ...
  done

EXT-ORACLE (slow, takes a few minutes) ...
  computing for TRAIN split (18324 rows) ...
  ext-oracle: 2000/18324
  ext-oracle: 4000/18324
  ext-oracle: 6000/18324
  ext-oracle: 8000/18324
  ext-oracle: 10000/18324
  ext-oracle: 12000/18324
  ext-oracle: 14000/18324
  ext-oracle: 16000/18324
  ext-oracle: 18000/18324
  train done.
  computing for DEV split (3927 rows) ...
  ext-oracle: 2000/3927
  dev done.
  computing for TEST split (3927 rows) ...
  ext-oracle: 2000/3927
  test done.

All features computed.


## 9. Classifiers and feature-set config

MLP uses early stopping: holds out 10% for validation, stops after 5 epochs with no improvement.

In [ ]:
def make_models():
    return {
        "LR":  LogisticRegression(max_iter=2000, C=1.0, solver="liblinear"),
        "SVM": LinearSVC(C=1.0),
        "MLP": MLPClassifier(hidden_layer_sizes=(128,), max_iter=100,
                             early_stopping=True, validation_fraction=0.1,
                             n_iter_no_change=5, random_state=SEED),
    }

feature_sets = {
    "no_features": [],
    "cosine":      ["cos"],
    "3features":   ["cos", "lead1", "nov1"],
    "5features":   ["cos", "lead1", "ext", "nov1", "nov2"],
}

def stack(fs, X, cos, l1, ex, n1, n2):
    parts = []
    if "cos"   in fs: parts.append(cos.reshape(-1, 1))
    if "lead1" in fs: parts.append(l1.reshape(-1, 1))
    if "ext"   in fs: parts.append(ex.reshape(-1, 1))
    if "nov1"  in fs: parts.append(n1.reshape(-1, 1))
    if "nov2"  in fs: parts.append(n2.reshape(-1, 1))
    if not parts:
        return X
    return hstack([X, csr_matrix(np.hstack(parts))]).tocsr()

## 10. 5-fold cross-validation on train

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []

for fold, (tr, va) in enumerate(skf.split(train_df, y_train), 1):
    print(f"\n--- fold {fold} ---")

    av, hv, sv = make_tfidf(), make_tfidf(), make_tfidf()
    Xtr = hstack([av.fit_transform(train_df["article"].iloc[tr]),
                  hv.fit_transform(train_df["headline"].iloc[tr])]).tocsr()
    Xva = hstack([av.transform(train_df["article"].iloc[va]),
                  hv.transform(train_df["headline"].iloc[va])]).tocsr()
    sv.fit(pd.concat([train_df["article"].iloc[tr], train_df["headline"].iloc[tr]]))

    cos_tr = cosine_sim(train_df["article"].iloc[tr], train_df["headline"].iloc[tr], sv)
    cos_va = cosine_sim(train_df["article"].iloc[va], train_df["headline"].iloc[va], sv)

    ytr, yva = y_train[tr], y_train[va]

    for fs_name, fs in feature_sets.items():
        Xtr_f = stack(fs, Xtr, cos_tr, lead1_train[tr], ext_train[tr], nov1_train[tr], nov2_train[tr])
        Xva_f = stack(fs, Xva, cos_va, lead1_train[va], ext_train[va], nov1_train[va], nov2_train[va])

        for name, clf in make_models().items():
            clf.fit(Xtr_f, ytr)
            yp = clf.predict(Xva_f)
            per_class = f1_score(yva, yp, labels=[0, 1, 2], average=None)
            w = f1_score(yva, yp, average="weighted")
            m = f1_score(yva, yp, average="macro")
            cv_results.append({
                "feature_set": fs_name, "model": name, "fold": fold,
                "HREL": per_class[0], "MREL": per_class[1], "LREL": per_class[2],
                "f1_weighted": w, "f1_macro": m,
            })
            print(f"  {fs_name:12s} {name:3s}   "
                  f"HREL={per_class[0]:.3f}  MREL={per_class[1]:.3f}  LREL={per_class[2]:.3f}   "
                  f"w={w:.3f}  m={m:.3f}")

cv_results = pd.DataFrame(cv_results)


--- fold 1 ---
  no_features  LR    HREL=0.550  MREL=0.496  LREL=0.594   w=0.542  m=0.547
  no_features  SVM   HREL=0.513  MREL=0.468  LREL=0.541   w=0.504  m=0.507
  no_features  MLP   HREL=0.593  MREL=0.448  LREL=0.599   w=0.539  m=0.547
  cosine       LR    HREL=0.570  MREL=0.506  LREL=0.597   w=0.553  m=0.558
  cosine       SVM   HREL=0.527  MREL=0.471  LREL=0.550   w=0.512  m=0.516
  cosine       MLP   HREL=0.600  MREL=0.471  LREL=0.584   w=0.546  m=0.552
  3features    LR    HREL=0.591  MREL=0.518  LREL=0.586   w=0.561  m=0.565
  3features    SVM   HREL=0.556  MREL=0.485  LREL=0.539   w=0.524  m=0.527
  3features    MLP   HREL=0.618  MREL=0.501  LREL=0.597   w=0.566  m=0.572
  5features    LR    HREL=0.593  MREL=0.517  LREL=0.589   w=0.562  m=0.566
  5features    SVM   HREL=0.557  MREL=0.485  LREL=0.540   w=0.524  m=0.527
  5features    MLP   HREL=0.576  MREL=0.523  LREL=0.605   w=0.564  m=0.568

--- fold 2 ---
  no_features  LR    HREL=0.568  MREL=0.495  LREL=0.598   w=0.548  m

## 11. Evaluate on dev and test

In [ ]:
eval_results = []

for fs_name, fs in feature_sets.items():
    print(f"\n--- {fs_name} ---")
    Xtr = stack(fs, X_train, cos_train, lead1_train, ext_train, nov1_train, nov2_train)
    Xdv = stack(fs, X_dev,   cos_dev,   lead1_dev,   ext_dev,   nov1_dev,   nov2_dev)
    Xte = stack(fs, X_test,  cos_test,  lead1_test,  ext_test,  nov1_test,  nov2_test)

    for name, clf in make_models().items():
        clf.fit(Xtr, y_train)

        yp_dev  = clf.predict(Xdv)
        yp_test = clf.predict(Xte)

        pc_dev  = f1_score(y_dev,  yp_dev,  labels=[0, 1, 2], average=None)
        pc_test = f1_score(y_test, yp_test, labels=[0, 1, 2], average=None)

        w_dev,  m_dev  = f1_score(y_dev,  yp_dev,  average="weighted"), f1_score(y_dev,  yp_dev,  average="macro")
        w_test, m_test = f1_score(y_test, yp_test, average="weighted"), f1_score(y_test, yp_test, average="macro")

        eval_results.append({
            "feature_set": fs_name, "model": name,
            "dev_HREL":  pc_dev[0],  "dev_MREL":  pc_dev[1],  "dev_LREL":  pc_dev[2],
            "dev_w":  round(w_dev,  3), "dev_m":  round(m_dev,  3),
            "test_HREL": pc_test[0], "test_MREL": pc_test[1], "test_LREL": pc_test[2],
            "test_w": round(w_test, 3), "test_m": round(m_test, 3),
        })
        print(f"  {name:3s}   "
              f"dev  HREL={pc_dev[0]:.2f} MREL={pc_dev[1]:.2f} LREL={pc_dev[2]:.2f}  w={w_dev:.2f} m={m_dev:.2f}    "
              f"test HREL={pc_test[0]:.2f} MREL={pc_test[1]:.2f} LREL={pc_test[2]:.2f}  w={w_test:.2f} m={m_test:.2f}")

eval_results = pd.DataFrame(eval_results)


--- no_features ---
  LR    dev  HREL=0.57 MREL=0.49 LREL=0.58  w=0.54 m=0.55    test HREL=0.56 MREL=0.50 LREL=0.60  w=0.55 m=0.55
  SVM   dev  HREL=0.53 MREL=0.45 LREL=0.54  w=0.50 m=0.50    test HREL=0.54 MREL=0.47 LREL=0.54  w=0.51 m=0.51
  MLP   dev  HREL=0.60 MREL=0.47 LREL=0.59  w=0.55 m=0.55    test HREL=0.59 MREL=0.49 LREL=0.61  w=0.55 m=0.56

--- cosine ---
  LR    dev  HREL=0.59 MREL=0.49 LREL=0.59  w=0.55 m=0.56    test HREL=0.58 MREL=0.51 LREL=0.59  w=0.56 m=0.56
  SVM   dev  HREL=0.54 MREL=0.46 LREL=0.54  w=0.51 m=0.51    test HREL=0.55 MREL=0.48 LREL=0.55  w=0.52 m=0.53
  MLP   dev  HREL=0.58 MREL=0.52 LREL=0.58  w=0.56 m=0.56    test HREL=0.57 MREL=0.53 LREL=0.59  w=0.56 m=0.56

--- 3features ---
  LR    dev  HREL=0.62 MREL=0.52 LREL=0.59  w=0.57 m=0.58    test HREL=0.60 MREL=0.53 LREL=0.60  w=0.57 m=0.58
  SVM   dev  HREL=0.57 MREL=0.47 LREL=0.55  w=0.52 m=0.53    test HREL=0.56 MREL=0.49 LREL=0.54  w=0.53 m=0.53
  MLP   dev  HREL=0.63 MREL=0.52 LREL=0.60  w=0.58 m=0.5

## 12. Summary

In [ ]:
FS_NAMES = {
    "no_features": "Without Feature Vector",
    "cosine":      "Cosine Similarity",
    "3features":   "[Cos, LEAD-1, Novel 1-gram %]",
    "5features":   "[Cos, LEAD-1, EXT-ORACLE, Novel 1-gram %, Novel 2-gram %]",
}

def print_paper_table(title, rows_by_fs_model):
    print(title)
    print("=" * 100)
    print(f"{'Feature Vector':<52} {'Classifier':<10}  {'HREL':>5} {'MREL':>5} {'LREL':>5}   "
          f"{'W':>5} {'M':>5}")
    print("-" * 100)
    for fs in ["no_features", "cosine", "3features", "5features"]:
        first = True
        for model in ["LR", "SVM", "MLP"]:
            r = rows_by_fs_model[(fs, model)]
            fs_label = FS_NAMES[fs] if first else ""
            print(f"{fs_label:<52} {model:<10}  "
                  f"{r['HREL']:.2f}  {r['MREL']:.2f}  {r['LREL']:.2f}    "
                  f"{r['W']:.2f}  {r['M']:.2f}")
            first = False
        print("-" * 100)


cv_avg = (cv_results
          .groupby(["feature_set", "model"])[["HREL", "MREL", "LREL", "f1_weighted", "f1_macro"]]
          .mean())

cv_rows = {}
for (fs, model), row in cv_avg.iterrows():
    cv_rows[(fs, model)] = {"HREL": row["HREL"], "MREL": row["MREL"], "LREL": row["LREL"],
                            "W": row["f1_weighted"], "M": row["f1_macro"]}

print_paper_table("Table 3 (our numbers): 5-fold CV averages on TRAIN. Compare to the paper.", cv_rows)


dev_rows, test_rows = {}, {}
for _, r in eval_results.iterrows():
    dev_rows[(r["feature_set"], r["model"])] = {"HREL": r["dev_HREL"], "MREL": r["dev_MREL"],
                                                "LREL": r["dev_LREL"], "W": r["dev_w"], "M": r["dev_m"]}
    test_rows[(r["feature_set"], r["model"])] = {"HREL": r["test_HREL"], "MREL": r["test_MREL"],
                                                 "LREL": r["test_LREL"], "W": r["test_w"], "M": r["test_m"]}

print()
print_paper_table("Same layout: DEV set", dev_rows)
print()
print_paper_table("Same layout: TEST set", test_rows)


print("\nPaper's Table 3 reference numbers to compare against")
print("=" * 100)
paper = {
    ("no_features", "LR"):  (0.57, 0.50, 0.59, 0.55, 0.55),
    ("no_features", "SVM"): (0.55, 0.49, 0.57, 0.53, 0.54),
    ("no_features", "MLP"): (0.55, 0.49, 0.58, 0.54, 0.54),
    ("cosine",      "LR"):  (0.58, 0.50, 0.59, 0.55, 0.56),
    ("cosine",      "SVM"): (0.56, 0.49, 0.58, 0.54, 0.54),
    ("cosine",      "MLP"): (0.56, 0.49, 0.56, 0.53, 0.54),
    ("3features",   "LR"):  (0.61, 0.53, 0.59, 0.58, 0.58),
    ("3features",   "SVM"): (0.60, 0.52, 0.58, 0.57, 0.57),
    ("3features",   "MLP"): (0.60, 0.54, 0.55, 0.56, 0.56),
    ("5features",   "LR"):  (0.62, 0.53, 0.59, 0.58, 0.58),
    ("5features",   "SVM"): (0.60, 0.52, 0.58, 0.57, 0.57),
    ("5features",   "MLP"): (0.60, 0.50, 0.61, 0.56, 0.57),
}
paper_rows = {k: {"HREL": v[0], "MREL": v[1], "LREL": v[2], "W": v[3], "M": v[4]}
              for k, v in paper.items()}
print_paper_table("", paper_rows)

Table 3 (our numbers): 5-fold CV averages on TRAIN. Compare to the paper.
Feature Vector                                       Classifier   HREL  MREL  LREL       W     M
----------------------------------------------------------------------------------------------------
Without Feature Vector                               LR          0.56  0.49  0.58    0.54  0.54
                                                     SVM         0.51  0.46  0.54    0.50  0.50
                                                     MLP         0.56  0.49  0.57    0.54  0.54
----------------------------------------------------------------------------------------------------
Cosine Similarity                                    LR          0.58  0.49  0.59    0.55  0.55
                                                     SVM         0.53  0.47  0.54    0.51  0.51
                                                     MLP         0.58  0.48  0.58    0.54  0.55
---------------------------------------------------

## 13. Reference: Table 3 in the paper

| feature set | LR | SVM | MLP |
|-------------|----|-----|-----|
| no_features | 0.55 | 0.53 | 0.54 |
| cosine      | 0.55 | 0.54 | 0.53 |
| 3features   | 0.58 | 0.57 | 0.56 |
| 5features   | 0.58 | 0.57 | 0.56 |

(Weighted F1, 5-fold CV on train.)